In [1]:
# ============================================================
# Dataset Quality Checker & Reporter
# Checks for: corrupted videos, class imbalance, video quality
# ============================================================

import cv2
from pathlib import Path
import numpy as np
from collections import Counter
import pandas as pd

# ================= CONFIG =================
DATASET_DIR = r"D:\Users\Anoshia\BattingEdge_FYP\dataset"
# ==========================================

print("="*70)
print("DATASET QUALITY CHECKER")
print("="*70)
print()

dataset_path = Path(DATASET_DIR)

if not dataset_path.exists():
    print(f"❌ Dataset not found at {DATASET_DIR}")
    exit(1)

all_stats = {}

for split in ["train", "val", "test"]:
    split_dir = dataset_path / split
    
    if not split_dir.exists():
        print(f"⚠️  {split} directory not found, skipping...")
        continue
    
    print(f"\n{'='*70}")
    print(f"CHECKING {split.upper()} SET")
    print(f"{'='*70}\n")
    
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    
    split_stats = {
        "total_videos": 0,
        "corrupted": [],
        "too_short": [],
        "low_fps": [],
        "class_counts": {},
        "duration_stats": [],
        "resolution_stats": [],
        "fps_stats": []
    }
    
    for cls in classes:
        class_dir = split_dir / cls
        video_files = list(class_dir.glob("*.mp4"))
        
        split_stats["class_counts"][cls] = len(video_files)
        split_stats["total_videos"] += len(video_files)
        
        print(f"📂 Checking {cls}... ({len(video_files)} videos)")
        
        for vid in video_files:
            cap = cv2.VideoCapture(str(vid))
            
            if not cap.isOpened():
                split_stats["corrupted"].append(str(vid.name))
                print(f"   ❌ CORRUPTED: {vid.name}")
                continue
            
            # Get video properties
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            duration = frame_count / fps if fps > 0 else 0
            
            split_stats["fps_stats"].append(fps)
            split_stats["duration_stats"].append(duration)
            split_stats["resolution_stats"].append((width, height))
            
            # Check for issues
            if frame_count < 30:  # Less than 1 second at 30fps
                split_stats["too_short"].append(f"{vid.name} ({frame_count} frames)")
                print(f"   ⚠️  TOO SHORT: {vid.name} - {frame_count} frames")
            
            if fps < 20:
                split_stats["low_fps"].append(f"{vid.name} ({fps:.1f} fps)")
                print(f"   ⚠️  LOW FPS: {vid.name} - {fps:.1f} fps")
            
            cap.release()
    
    all_stats[split] = split_stats
    
    # Print split summary
    print(f"\n📊 {split.upper()} SUMMARY:")
    print(f"   Total videos: {split_stats['total_videos']}")
    print(f"   Corrupted: {len(split_stats['corrupted'])}")
    print(f"   Too short (<30 frames): {len(split_stats['too_short'])}")
    print(f"   Low FPS (<20): {len(split_stats['low_fps'])}")
    
    if split_stats["duration_stats"]:
        print(f"   Avg duration: {np.mean(split_stats['duration_stats']):.2f}s")
        print(f"   Avg FPS: {np.mean(split_stats['fps_stats']):.1f}")
    
    print(f"\n   Class distribution:")
    for cls, count in split_stats["class_counts"].items():
        print(f"      {cls}: {count} videos")

# ================= OVERALL REPORT =================
print("\n" + "="*70)
print("OVERALL DATASET REPORT")
print("="*70)
print()

# Class balance check
print("📊 CLASS BALANCE ANALYSIS:")
print()

for split in ["train", "val", "test"]:
    if split in all_stats:
        counts = all_stats[split]["class_counts"]
        if counts:
            min_class = min(counts.values())
            max_class = max(counts.values())
            imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
            
            print(f"{split.upper()}:")
            print(f"   Min class: {min_class} videos")
            print(f"   Max class: {max_class} videos")
            print(f"   Imbalance ratio: {imbalance_ratio:.2f}x")
            
            if imbalance_ratio > 2:
                print(f"   ⚠️  WARNING: Significant class imbalance!")
            else:
                print(f"   ✅ Class balance is good")
            print()

# Quality issues summary
print("🔍 QUALITY ISSUES SUMMARY:")
print()

total_corrupted = sum(len(all_stats[s]["corrupted"]) for s in all_stats)
total_too_short = sum(len(all_stats[s]["too_short"]) for s in all_stats)
total_low_fps = sum(len(all_stats[s]["low_fps"]) for s in all_stats)
total_videos = sum(all_stats[s]["total_videos"] for s in all_stats)

print(f"Total videos: {total_videos}")
print(f"Corrupted: {total_corrupted} ({total_corrupted/total_videos*100:.1f}%)")
print(f"Too short: {total_too_short} ({total_too_short/total_videos*100:.1f}%)")
print(f"Low FPS: {total_low_fps} ({total_low_fps/total_videos*100:.1f}%)")
print()

if total_corrupted + total_too_short + total_low_fps == 0:
    print("✅ EXCELLENT! No quality issues detected!")
elif (total_corrupted + total_too_short + total_low_fps) / total_videos < 0.05:
    print("✅ GOOD! Less than 5% problematic videos.")
else:
    print("⚠️  ATTENTION NEEDED! Consider removing/replacing problematic videos.")

print("\n" + "="*70)
print("💡 RECOMMENDATION:")
print()

needs_augmentation = False

for split in all_stats:
    counts = all_stats[split]["class_counts"]
    if counts:
        min_count = min(counts.values())
        if min_count < 50:
            needs_augmentation = True
            print(f"⚠️  {split.upper()}: Minimum class has only {min_count} videos")

if needs_augmentation:
    print("\n   Consider data augmentation to balance classes.")
    print("   Run the augmentation script next.")
else:
    print("✅ Dataset size is sufficient for training.")
    print("   No augmentation needed.")

print("="*70)

DATASET QUALITY CHECKER


CHECKING TRAIN SET

📂 Checking Cover Drive... (617 videos)
   ⚠️  TOO SHORT: aug_speed_up_134_cover (473).mp4 - 26 frames
   ⚠️  TOO SHORT: aug_speed_up_14_cover (232).mp4 - 19 frames
   ⚠️  TOO SHORT: aug_speed_up_199_cover (229).mp4 - 28 frames
   ⚠️  TOO SHORT: aug_speed_up_217_cover (104).mp4 - 27 frames
   ⚠️  TOO SHORT: aug_speed_up_286_cover (104).mp4 - 24 frames
   ⚠️  TOO SHORT: aug_speed_up_343_cover (148).mp4 - 26 frames
   ⚠️  TOO SHORT: aug_speed_up_360_cover (442).mp4 - 22 frames
   ⚠️  TOO SHORT: aug_speed_up_366_cover (232).mp4 - 19 frames
   ⚠️  TOO SHORT: aug_speed_up_41_cover (148).mp4 - 26 frames
   ⚠️  TOO SHORT: aug_speed_up_439_cover (117).mp4 - 25 frames
   ⚠️  TOO SHORT: aug_speed_up_446_cover (157).mp4 - 28 frames
   ⚠️  TOO SHORT: aug_speed_up_549_cover (113).mp4 - 29 frames
   ⚠️  TOO SHORT: cover (146).mp4 - 27 frames
   ⚠️  TOO SHORT: cover (232).mp4 - 27 frames
   ⚠️  TOO SHORT: cover (26).mp4 - 23 frames
   ⚠️  TOO SHORT: cover 